In [1]:

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
#Serve para ler o arquivo CSV e transforma a coluna Data em data
df = pd.read_csv("../dados/mecaniqa_dataset.csv", parse_dates=["Data"])

df = df.sort_values("Data")
df = df.set_index("Data")

print("Tamanho dos dados:", df.shape)
print("Colunas:", df.columns.tolist())

display(df.head())


Tamanho dos dados: (731, 2)
Colunas: ['Trocas_Oleo', 'Manutencao_Motor']


,Trocas_Oleo,Manutencao_Motor
Data,,
2024-01-01,11.0,2.0
2024-01-02,9.0,4.0
2024-01-03,12.0,1.0
2024-01-04,15.0,8.0
2024-01-05,25.0,10.0


In [3]:

# Guarda uma cópia dos dados originais
dados = df.copy()
# Guarda a quantidade de trocas de óleo do dia anterior
dados["Lag_1"] = dados["Trocas_Oleo"].shift(1)
# Guarda a quantidade de trocas de óleo de 7 dias atrás
dados["Lag_7"] = dados["Trocas_Oleo"].shift(7)
# Calcula a média de trocas de óleo dos últimos 7 dias
dados["Media_7"] = dados["Trocas_Oleo"].shift(1).rolling(7).mean()
# Identifica o dia da semana
dados["Dia_Semana"] = dados.index.dayofweek

colunas = ["Lag_1", "Lag_7", "Media_7", "Dia_Semana"]
dados = dados.dropna(subset=colunas + ["Trocas_Oleo"])

X = dados[colunas]
y = dados["Trocas_Oleo"]

tscv = TimeSeriesSplit(n_splits=5)

print("Registros preparados:", len(dados))
print("Características:", colunas)
print("Divisões temporais:", tscv.n_splits)


Registros preparados: 711
Características: ['Lag_1', 'Lag_7', 'Media_7', 'Dia_Semana']
Divisões temporais: 5


In [5]:

# Modelos e valores que serão testados
modelos = {
    "Árvore de Decisão": (
        DecisionTreeRegressor(random_state=42),
        {
            "max_depth": [3, 5, 10, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    ),
    "Random Forest": (
        RandomForestRegressor(random_state=42),
        {
            "n_estimators": [50, 100, 200],
            "max_depth": [5, 10, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    )
}

resultados = []

for nome, (modelo, parametros) in modelos.items():
    print("\nTestando:", nome)

    # Grid Search
    grid = GridSearchCV(
        modelo,
        parametros,
        scoring="neg_mean_absolute_error",
        cv=tscv,
        n_jobs=-1
    )

    inicio = time.perf_counter()
    grid.fit(X, y)
    tempo_grid = time.perf_counter() - inicio

    resultados.append({
        "Modelo": nome,
        "Método": "Grid Search",
        "MAE": -grid.best_score_,
        "Tempo (s)": tempo_grid,
        "Melhores parâmetros": grid.best_params_
    })

    # Random Search
    random = RandomizedSearchCV(
        modelo,
        parametros,
        n_iter=10,
        scoring="neg_mean_absolute_error",
        cv=tscv,
        random_state=42,
        n_jobs=-1
    )

    inicio = time.perf_counter()
    random.fit(X, y)
    tempo_random = time.perf_counter() - inicio

    resultados.append({
        "Modelo": nome,
        "Método": "Random Search",
        "MAE": -random.best_score_,
        "Tempo (s)": tempo_random,
        "Melhores parâmetros": random.best_params_
    })

# Mostra a comparação dos resultados
tabela = pd.DataFrame(resultados)

display(
    tabela.sort_values("MAE")
    .reset_index(drop=True)
    .round({"MAE": 2, "Tempo (s)": 2})
)



Testando: Árvore de Decisão

Testando: Random Forest


,Modelo,Método,MAE,Tempo (s),Melhores parâmetros
0,Random Forest,Grid Search,4.44,21.73,"{'max_depth': 10, 'min_samples_leaf': 2, 'min_..."
1,Random Forest,Random Search,4.50,2.05,"{'n_estimators': 50, 'min_samples_split': 5, '..."
2,Árvore de Decisão,Grid Search,4.75,0.24,"{'max_depth': 5, 'min_samples_leaf': 1, 'min_s..."
3,Árvore de Decisão,Random Search,5.07,0.11,"{'min_samples_split': 5, 'min_samples_leaf': 4..."
